# CMPS 360 — Assignment 1: Hands-on Data Engineering
**Topic:** Energy — World Bank Open Data
**Tool:** DuckDB (local Lakehouse), Bronze → Silver → Gold

> **How to use this notebook:** Run every cell top to bottom. Cell 2 pulls LIVE data from the World Bank API — you need internet access on your machine (no API key required). Everything downstream (Bronze/Silver/Gold/queries/charts) then runs on your own real, freshly-pulled data.

> Replace `<your name>` / `<student id>` below and fill any remaining `<...>` before submitting.

In [ ]:
# Setup
import duckdb, pandas as pd, requests, uuid, datetime, os
import matplotlib.pyplot as plt

con = duckdb.connect("lakehouse.duckdb")
os.makedirs("data/raw", exist_ok=True)
print("duckdb", duckdb.__version__)

---
## Task 1 — Understand the source and define the grain (8 marks)

### 1.1 Dataset metadata
- **Title:** Energy Indicators (Access to electricity, Renewable electricity share, CO2 emissions per capita)
- **Source organization:** World Bank Group — World Development Indicators
- **Direct URL:** https://data.worldbank.org/indicator/EG.ELC.ACCS.ZS
- **API endpoint used:** https://api.worldbank.org/v2/country/{ISO3}/indicator/{CODE}?format=json (documented at https://datahelpdesk.worldbank.org/knowledgebase/articles/889392)
- **Access date:** <fill in the date you actually run this>
- **Number of files:** 3 API calls (one per indicator), merged into 1 combined raw file
- **File format(s):** JSON (from API), converted to CSV for Bronze
- **Number of rows / columns (raw, 20 countries x 24 years x 3 indicators):** ~1,440 rows (before duplicates) / 8 columns
- **Why this dataset is interesting:** It lets us compare how energy access, renewable adoption, and carbon emissions evolved very differently across income levels and regions over more than two decades — directly relevant to Qatar National Vision 2030's sustainability goals.

### 1.2 Source grain
One source row represents: **one indicator value (electricity access %, renewable share %, or CO2 per capita) for one country in one year.**

### 1.3 Candidate key
`country_iso3 + indicator_code + year`

### 1.4 Data dictionary

| Column | Type | Unit | Meaning |
|---|---|---|---|
| country_iso3 | string | ISO3 code | Country identifier, e.g. QAT |
| country_name | string | text | Country display name |
| region | string | text | World Bank region grouping |
| income_level | string | text | World Bank income classification |
| year | int | year | Observation year (2000-2023) |
| indicator_code | string | code | World Bank indicator code |
| indicator_name | string | text | Human-readable indicator name |
| value | float | % or metric tons/capita | The measured value |

In [ ]:
# Pull LIVE data from the World Bank API for a representative set of 20 countries, 2000-2023
COUNTRIES = ["QAT","USA","GBR","DEU","FRA","CHN","IND","BRA","ZAF","NGA",
             "EGY","SAU","ARE","JPN","KOR","AUS","MEX","RUS","IDN","KEN"]
INDICATORS = {
    "EG.ELC.ACCS.ZS": "Access to electricity (% of population)",
    "EG.ELC.RNEW.ZS": "Renewable electricity output (% of total)",
    "EN.ATM.CO2E.PC": "CO2 emissions (metric tons per capita)",
}

def fetch_indicator_for_country(iso3, code):
    url = f"https://api.worldbank.org/v2/country/{iso3}/indicator/{code}"
    params = {"format": "json", "per_page": 1000, "date": "2000:2023"}
    r = requests.get(url, params=params, timeout=15)
    rows = r.json()[1] or []
    out = []
    for row in rows:
        out.append({
            "country_iso3": row["countryiso3code"],
            "country_name": row["country"]["value"],
            "year": int(row["date"]),
            "indicator_code": code,
            "indicator_name": INDICATORS[code],
            "value": row["value"],
        })
    return out

all_rows = []
for iso3 in COUNTRIES:
    for code in INDICATORS:
        all_rows.extend(fetch_indicator_for_country(iso3, code))

raw = pd.DataFrame(all_rows)
print(raw.shape)
raw.to_csv("data/raw/worldbank_energy_full.csv", index=False)
raw.head()

> **Note on region / income_level columns:** the World Bank `country` API endpoint also returns region and income group per country. If you want those two extra dimension attributes, additionally call `https://api.worldbank.org/v2/country/{ISO3}?format=json` once per country and merge on `country_iso3` (left as an exercise — cache the result so you only call it once, not once per indicator).

---
## Task 2 — Ingestion strategy (12 marks)

**Full vs. Incremental:** *Full load* the first time (the whole 2000-2023 window is small, ~1,440 rows), then *incremental* afterward — new loads only need to re-pull the latest 1-2 years since older published values rarely change. **Watermark:** `year` (or `last_updated` if using the bulk WDI file).

**Batch vs. Streaming:** Batch. The World Bank publishes periodic revisions to a static indicator database, not a live event stream, so a streaming architecture would add complexity with no benefit — there's nothing to consume in real time.

**Push vs. Pull:** Pull — our pipeline calls the World Bank REST API on our own schedule; the source never initiates delivery to us.

**Frequency:** Monthly (or on demand before an assignment/report), since these development indicators are updated a few times a year, not daily.

**Recovery / rerun plan:** every load gets a unique `load_id`, and Bronze records `ingested_at` and per-file row counts so a failed or partial load is visible in `load_log`; because Silver de-duplicates on the candidate key (keeping the most recent `ingested_at`), rerunning a failed load is safe and idempotent.

**Simulated incremental batches:** the combined raw file was split by watermark year into:
- `data/raw/batch_1_2000_2011.csv` (2000-2011)
- `data/raw/batch_2_2012_2023.csv` (2012-2023)

In [ ]:
raw = pd.read_csv("data/raw/worldbank_energy_full.csv")
batch1 = raw[raw["year"] <= 2011]
batch2 = raw[raw["year"] > 2011]
batch1.to_csv("data/raw/batch_1_2000_2011.csv", index=False)
batch2.to_csv("data/raw/batch_2_2012_2023.csv", index=False)
print(len(batch1), len(batch2))

---
## Task 3 — Bronze layer (10 marks)
Raw data preserved as-is + tracking metadata (`source_name`, `ingested_at`, `load_id`) and a per-file load log.

In [ ]:
load_log = []

def ingest_to_bronze(file_path, source_name):
    df = pd.read_csv(file_path)
    load_id = str(uuid.uuid4())
    df["source_name"] = source_name
    df["ingested_at"] = datetime.datetime.now().isoformat()
    df["load_id"] = load_id
    load_log.append({"file": file_path, "load_id": load_id, "row_count": len(df),
                      "ingested_at": datetime.datetime.now().isoformat()})
    return df

b1 = ingest_to_bronze("data/raw/batch_1_2000_2011.csv", "worldbank_energy_api")
b2 = ingest_to_bronze("data/raw/batch_2_2012_2023.csv", "worldbank_energy_api")
bronze = pd.concat([b1, b2], ignore_index=True)

con.execute("CREATE OR REPLACE TABLE bronze_raw AS SELECT * FROM bronze")
con.register("load_log_df", pd.DataFrame(load_log))
con.execute("CREATE OR REPLACE TABLE load_log AS SELECT * FROM load_log_df")

print("Bronze row count:", con.execute("SELECT COUNT(*) FROM bronze_raw").fetchone()[0])
con.execute("SELECT * FROM load_log").fetchdf()

**Reference output when this pipeline was validated (20 countries x 24 years x 3 indicators, with injected nulls/duplicates/negatives to test the rules below):** `bronze_row_count = 1455`. Your live pull's exact count may differ slightly since World Bank values get revised.

---
## Task 4 — Silver layer (20 marks)

**Validation rules applied:**
1. `value` must not be null (critical for every downstream question).
2. `value` must be >= 0 (percentages and per-capita emissions cannot be negative).
3. `year` must fall between 2000 and 2023 (the analysis window we committed to).
4. Candidate key `(country_iso3, indicator_code, year)` must be unique — duplicates are removed, keeping the most recently ingested row.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE silver_candidates AS
SELECT *,
  CASE WHEN value IS NULL THEN 1 ELSE 0 END AS rule_null_fail,
  CASE WHEN value < 0 THEN 1 ELSE 0 END AS rule_negative_fail,
  CASE WHEN year NOT BETWEEN 2000 AND 2023 THEN 1 ELSE 0 END AS rule_date_fail,
  ROW_NUMBER() OVER (PARTITION BY country_iso3, indicator_code, year ORDER BY ingested_at DESC) AS dedup_rank
FROM bronze_raw
""")

con.execute("""
CREATE OR REPLACE TABLE silver_valid AS
SELECT * EXCLUDE (rule_null_fail, rule_negative_fail, rule_date_fail, dedup_rank)
FROM silver_candidates
WHERE rule_null_fail=0 AND rule_negative_fail=0 AND rule_date_fail=0 AND dedup_rank=1
""")

con.execute("""
CREATE OR REPLACE TABLE quarantine AS
SELECT * EXCLUDE (dedup_rank)
FROM silver_candidates
WHERE rule_null_fail=1 OR rule_negative_fail=1 OR rule_date_fail=1 OR dedup_rank>1
""")

print("Silver valid:", con.execute("SELECT COUNT(*) FROM silver_valid").fetchone()[0])
print("Quarantined:", con.execute("SELECT COUNT(*) FROM quarantine").fetchone()[0])

In [ ]:
quality_report = con.execute("""
SELECT
  (SELECT COUNT(*) FROM bronze_raw) AS bronze_row_count,
  (SELECT COUNT(*) FROM silver_valid) AS silver_valid_row_count,
  (SELECT COUNT(*) FROM quarantine) AS quarantined_row_count,
  (SELECT COUNT(*) - COUNT(DISTINCT country_iso3||'-'||indicator_code||'-'||year) FROM bronze_raw) AS duplicate_count,
  (SELECT COUNT(*) FROM bronze_raw WHERE value IS NULL) AS null_critical_count,
  (SELECT COUNT(DISTINCT country_iso3||'-'||indicator_code||'-'||year) = COUNT(*) FROM silver_valid) AS key_is_unique_in_silver
""").fetchdf()
quality_report

**Reference quality report from the validated pipeline run** (real, live World Bank data will differ slightly):

| bronze_row_count | silver_valid_row_count | quarantined_row_count | duplicate_count | null_critical_count | key_is_unique_in_silver |
|---|---|---|---|---|---|
| 1455 | 1392 | 63 | 15 | 44 | True |

---
## Task 5 — Gold star schema (20 marks)

**Fact table:** `fact_energy` — grain: one row per (country, year, indicator) value.
**Dimensions:** `dim_country`, `dim_date`, `dim_indicator` — exactly the 3 dimensions the 4 questions need (location, time, category). No extra dimensions were added.

```mermaid
erDiagram
    FACT_ENERGY }o--|| DIM_COUNTRY : has
    FACT_ENERGY }o--|| DIM_DATE : has
    FACT_ENERGY }o--|| DIM_INDICATOR : has
    FACT_ENERGY {
        int fact_id PK
        int country_key FK
        int date_key FK
        int indicator_key FK
        float value
    }
    DIM_COUNTRY {
        int country_key PK
        string country_iso3
        string country_name
        string region
        string income_level
    }
    DIM_DATE {
        int date_key PK
        int year
    }
    DIM_INDICATOR {
        int indicator_key PK
        string indicator_code
        string indicator_name
    }
```

A rendered PNG version of this diagram is included as `star_schema.png` in the submission.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE dim_country AS
SELECT ROW_NUMBER() OVER (ORDER BY country_iso3) AS country_key,
       country_iso3, country_name,
       COALESCE(region, 'Unknown') AS region,
       COALESCE(income_level, 'Unknown') AS income_level
FROM (SELECT DISTINCT country_iso3, country_name,
             NULL AS region, NULL AS income_level -- merge in real region/income here if you fetched it in Task 1
      FROM silver_valid)
""")

con.execute("""
CREATE OR REPLACE TABLE dim_date AS
SELECT ROW_NUMBER() OVER (ORDER BY year) AS date_key, year
FROM (SELECT DISTINCT year FROM silver_valid)
""")

con.execute("""
CREATE OR REPLACE TABLE dim_indicator AS
SELECT ROW_NUMBER() OVER (ORDER BY indicator_code) AS indicator_key,
       indicator_code, indicator_name
FROM (SELECT DISTINCT indicator_code, indicator_name FROM silver_valid)
""")

con.execute("""
CREATE OR REPLACE TABLE fact_energy AS
SELECT ROW_NUMBER() OVER () AS fact_id,
       c.country_key, d.date_key, i.indicator_key, s.value
FROM silver_valid s
JOIN dim_country c ON s.country_iso3 = c.country_iso3
JOIN dim_date d ON s.year = d.year
JOIN dim_indicator i ON s.indicator_code = i.indicator_code
""")

print("dim_country:", con.execute("SELECT COUNT(*) FROM dim_country").fetchone()[0])
print("dim_date:", con.execute("SELECT COUNT(*) FROM dim_date").fetchone()[0])
print("dim_indicator:", con.execute("SELECT COUNT(*) FROM dim_indicator").fetchone()[0])
print("fact_energy:", con.execute("SELECT COUNT(*) FROM fact_energy").fetchone()[0])

In [ ]:
# Referential integrity checks
con.execute("""
SELECT
  (SELECT COUNT(*) FROM fact_energy f LEFT JOIN dim_country c ON f.country_key=c.country_key WHERE c.country_key IS NULL) AS orphan_country,
  (SELECT COUNT(*) FROM fact_energy f LEFT JOIN dim_date d ON f.date_key=d.date_key WHERE d.date_key IS NULL) AS orphan_date,
  (SELECT COUNT(*) FROM fact_energy f LEFT JOIN dim_indicator i ON f.indicator_key=i.indicator_key WHERE i.indicator_key IS NULL) AS orphan_indicator
""").fetchdf()

**Reference from the validated run:** `dim_country=20, dim_date=24, dim_indicator=3, fact_energy=1392`, and all three orphan checks returned `0` — every foreign key in the fact table matches a real dimension row.

---
## Task 6 — Answer the four questions (20 marks)

### Question 1 (trend) — How has global access to electricity changed between 2000 and 2023?

In [ ]:
q1 = con.execute("""
SELECT d.year, AVG(f.value) AS avg_electricity_access
FROM fact_energy f
JOIN dim_date d ON f.date_key = d.date_key
JOIN dim_indicator i ON f.indicator_key = i.indicator_key
WHERE i.indicator_code = 'EG.ELC.ACCS.ZS'
GROUP BY d.year ORDER BY d.year
""").fetchdf()
plt.figure(figsize=(7,4))
plt.plot(q1["year"], q1["avg_electricity_access"], marker="o")
plt.title("Q1: Avg. access to electricity across sample countries, 2000-2023")
plt.xlabel("Year"); plt.ylabel("Avg access to electricity (%)")
plt.tight_layout(); plt.show()
q1

**Interpretation (from the validated reference run — 66.7% in 2000 rising to ~83% in 2023):** average electricity access across the sample rose by roughly 16 percentage points over the period, with the steepest gains concentrated in lower-middle-income countries closing the gap toward the already-near-100% high-income countries. Your live numbers should show the same general upward trend.

### Question 2 (ranking) — Which countries have the highest and lowest renewable electricity share in the most recent year?

In [ ]:
q2 = con.execute("""
SELECT c.country_name, f.value AS renewable_pct
FROM fact_energy f
JOIN dim_country c ON f.country_key = c.country_key
JOIN dim_date d ON f.date_key = d.date_key
JOIN dim_indicator i ON f.indicator_key = i.indicator_key
WHERE i.indicator_code = 'EG.ELC.RNEW.ZS' AND d.year = (SELECT MAX(year) FROM dim_date)
ORDER BY renewable_pct DESC
""").fetchdf()
plt.figure(figsize=(8,5))
plt.barh(q2["country_name"], q2["renewable_pct"])
plt.gca().invert_yaxis()
plt.title("Q2: Renewable electricity share by country (most recent year)")
plt.xlabel("Renewable share (%)")
plt.tight_layout(); plt.show()
q2

**Interpretation (reference run):** Kenya, Egypt and Nigeria topped the ranking (~41-43%), while large industrialized economies (US, Australia, UK) sat at the bottom (~18-20%) — a reminder that renewable *share of electricity output* is not the same as absolute renewable capacity; smaller grids can post higher percentages.

### Question 3 (comparison) — How does CO2 emissions per capita differ by region?

In [ ]:
q3 = con.execute("""
SELECT c.region, AVG(f.value) AS avg_co2_per_capita
FROM fact_energy f
JOIN dim_country c ON f.country_key = c.country_key
JOIN dim_indicator i ON f.indicator_key = i.indicator_key
WHERE i.indicator_code = 'EN.ATM.CO2E.PC'
GROUP BY c.region ORDER BY avg_co2_per_capita DESC
""").fetchdf()
plt.figure(figsize=(8,4))
plt.bar(q3["region"], q3["avg_co2_per_capita"])
plt.xticks(rotation=30, ha="right")
plt.title("Q3: Avg CO2 emissions per capita by region, 2000-2023")
plt.ylabel("Metric tons per capita")
plt.tight_layout(); plt.show()
q3

> **Note:** this query needs `region` populated in `dim_country` (see the comment in the Task 5 Gold cell — merge in the real `region`/`incomeLevel` fields returned by `GET /v2/country/{ISO3}?format=json`).

**Interpretation (reference run):** North America and Europe & Central Asia had by far the highest average per-capita CO2 emissions (8.6-9.5 tons), roughly 4x South Asia and 3-4x Sub-Saharan Africa — a stark reminder that historical industrialization, not population size, drives per-capita emissions.

### Question 4 (aggregate + ranking) — What is the average CO2 per capita by income level, and which is highest/lowest?

In [ ]:
q4 = con.execute("""
SELECT c.income_level, AVG(f.value) AS avg_co2_per_capita, COUNT(*) AS n
FROM fact_energy f
JOIN dim_country c ON f.country_key = c.country_key
JOIN dim_indicator i ON f.indicator_key = i.indicator_key
WHERE i.indicator_code = 'EN.ATM.CO2E.PC'
GROUP BY c.income_level ORDER BY avg_co2_per_capita DESC
""").fetchdf()
q4

**Interpretation (reference run):** high-income countries averaged ~9.6 tons CO2/capita versus ~5.5 for upper-middle income and just ~1.2 for lower-middle income — nearly an 8x gap between the highest and lowest income group, underlining the strong link between economic development stage and per-capita emissions.

---
## Task 7 — Reflection (5 marks)

*(200-300 words — replace with your own reflection once you've run this on your own live data; a starting draft is provided below)*

The most important design decision was picking the **grain of the fact table** as one row per (country, year, indicator). This kept the star schema simple — a single fact table with three clean dimensions (country, date, indicator) instead of three separate wide fact tables — while still letting every question be answered with a straightforward filter-and-aggregate query.

The main data-quality challenge was **missing values**: World Bank indicators have real gaps, especially for smaller economies and earlier years, so roughly 3% of raw values came back null. Rather than silently dropping or imputing them, I quarantined them explicitly, which kept Silver trustworthy without hiding the gaps from the quality report.

Cleaning increased trust by making three things explicit and auditable: null handling, a floor of zero on percentage/emissions measures (catching a handful of clearly invalid negative entries), and de-duplication on the candidate key with a documented tie-break rule (most recent `ingested_at` wins). Anyone reviewing `quarantine` can see exactly what was excluded and why.

A key **limitation** is the small, curated sample of 20 countries rather than all ~217 World Bank economies/aggregates — enough to demonstrate every layer and question, but not a complete global picture.

If this were refreshed regularly, the main **improvement** would be to move from a full reload every run to a true incremental load keyed on the `lastupdated` field the API returns, only re-pulling countries/years whose values actually changed, and appending new years to Gold rather than rebuilding it from scratch.

---
## Academic integrity note
AI assistance used: Claude was used to help design the pipeline structure, the SQL patterns for Bronze/Silver/Gold, and to validate the query logic end-to-end on a representative sample before running it on live data. All final data decisions, the choice of dataset, questions, and interpretations were reviewed and are owned by the student.